In [0]:
df = spark.table("novacart_dev.bronze.orders")

In [0]:
from pyspark.sql import functions as F
df = df.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])

In [0]:
df = df.replace(["\\N", "?", "", "null", "NULL"], None)

In [0]:
string_cols = [c for c, t in df.dtypes if t == "string"]

for col in string_cols:
    df = df.withColumn(col, F.trim(F.lower(F.col(col))))

In [0]:
df = df.dropDuplicates(["order_id"])

In [0]:
df = df.withColumn(
    "order_date",
    F.coalesce(
        F.to_date(F.expr("try_to_timestamp(order_date, 'yyyy-MM-dd')")),
        F.to_date(F.expr("try_to_timestamp(order_date, 'yyyy/MM/dd')")),
        F.to_date(F.expr("try_to_timestamp(order_date, 'dd/MM/yyyy')"))
    )
)

In [0]:
status_map = {
    "completed": "completed",
    "complete": "completed",
    "done": "completed",
    "已完成": "completed",
    "abgeschlossen": "completed",
    "completado": "completed",
    "पूर्ण": "completed",


    "भेज दिया": "shipped",
    "versandt": "shipped",
    "已发货": "shipped",
    "enviado": "shipped",


    "pending": "pending",
    "in progress": "pending",
    "pendiente": "pending",
    "待处理": "pending",
    "ausstehend": "pending",
    "लंबित": "pending",

    "cancelled": "cancelled",
    "canceled": "cancelled",
    "रद्द": "cancelled",
    "storniert": "cancelled",
    "cancelado":"cancelled",
    "已取消": "cancelled",

    "returned": "returned"
}

mapping_expr = F.create_map([F.lit(x) for x in sum(status_map.items(), ())])

df = df.withColumn(
    "order_status_clean",
    mapping_expr.getItem(F.col("order_status"))
)

In [0]:
df = df.withColumn(
    "channel",
    F.when(F.col("channel").isin("web", "mobile"), F.col("channel"))
     .otherwise(None)
)

In [0]:
df = df.withColumn("total_amount", F.col("total_amount").cast("double"))

df = df.withColumn(
    "total_amount",
    F.when(F.col("total_amount") < 0, None).otherwise(F.col("total_amount"))
)

In [0]:
df = df.withColumn("country_code", F.upper(F.col("country_code")))

In [0]:
customers_df = spark.table("novacart_dev.silver.slv_customers")

df = df.join(
    customers_df.select("customer_id"),
    on="customer_id",
    how="left"
)

df = df.withColumn(
    "dq_orphan_customer",
    F.when(F.col("customer_id").isNull(), 1).otherwise(0)
)

In [0]:
df = df.withColumn("currency", F.upper(F.col("currency")))

In [0]:
df = df.withColumn(
    "dq_missing_order_id",
    F.when(F.col("order_id").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_order_date",
    F.when(F.col("order_date").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_status",
    F.when(F.col("order_status_clean").isNull(), 1).otherwise(0)
)

df = df.withColumn(
    "dq_invalid_amount",
    F.when(F.col("total_amount").isNull(), 1).otherwise(0)
)

In [0]:
df = df.withColumn("load_timestamp", F.current_timestamp())

In [0]:
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("novacart_dev.silver.slv_orders")

In [0]:
display(spark.table("novacart_dev.silver.slv_orders"))